# 记忆巩固（Memory Consolidation）

> 定期审查、合并、去重并强化记忆。灵感来自大脑在睡眠中的巩固机制。

想一想每个月底清理一个乱糟糟的文件柜。你会发现三个文件夹讲的是同一个项目，其中两个还记着互相矛盾的笔记。你把最好的部分合并成一个干净的文件夹，扔掉重复的。没有这种清理，文件柜很快就没法用了。

人脑在睡眠时运行类似的过程。它重放最近的经历，强化重要的部分，修剪重复项，并把新信息织入已有知识。正是这个过程把脆弱的短期痕迹变成牢固的长期记忆。

智能体记忆系统面对同样的问题，而且规模膨胀得更快。几百次交互之后，记忆库里堆满重复项：用户可能在五次不同的对话里提到自己的职位。矛盾也会混进来：用户三月说自己偏好 Python，六月却换成了 Rust。不清理，检索质量就会下降。

**记忆巩固**是智能体记忆库的后台维护过程。它审查已存的记忆，聚类相关条目，合并重复项，解决矛盾。结果是一个更干净的记忆库，随时间变好而不是变差。

本 notebook 教你用 **OpenAI SDK** 从零搭建一个巩固流程。你将实现聚类、去重、矛盾解决和重要性打分。

**读完本 notebook 你会理解：**
- 如何按语义相似度聚类相关记忆。
- 如何用 LLM 合并重复项、解决矛盾。
- 如何给记忆的重要性打分并修剪低价值条目。
- 什么时候巩固有帮助，什么时候会出问题。


## 核心概念

- **嵌入（embedding）**：一串数字（即向量），捕捉一段文本的含义。含义相近的文本会生成相近的嵌入。我们用嵌入衡量两条记忆的关联程度。
- **余弦相似度（cosine similarity）**：一个介于 -1 到 1 之间的分数，衡量两个嵌入方向的接近程度。我们把 0.8 以上视为「相关」，0.92 以上视为「近似重复」。
- **巩固触发条件（consolidation trigger）**：启动一轮巩固的条件。常见触发条件包括记忆库大小阈值、时间间隔或手动命令。
- **记忆合并（memory merging）**：把关于同一事实的两条或多条记忆合成一条更丰富的条目。例如：把「用户喜欢意大利菜」和「用户最爱的菜系是意大利菜，尤其喜欢意面」合并成一条。
- **去重（deduplication）**：找出并删除几乎相同的记忆，保留最完整或最新的版本。
- **矛盾解决（conflict resolution）**：处理相互矛盾记忆的策略。可选方案包括新近度优先（最新版本获胜）、来源优先（用户直接陈述的事实胜过推断）或 LLM 裁决（让模型来决定）。
- **重要性打分（importance scoring）**：巩固后重新计算每条记忆的价值。考虑因素包括访问频率、新近度和来源可靠性。


## 架构

<p align="center">
  <img src="../../images/diagrams/14_memory_consolidation.svg" alt="记忆巩固架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    MS["Memory Store\n(pre-consolidation)"] --> CT["Consolidation\nTrigger"]
    CT --> CE["Cluster Engine\n(similarity grouping)"]
    CE --> MD["Merge &\nDeduplicate"]
    MD --> CR["Conflict\nResolver"]
    CR --> IS["Importance\nScorer"]
    IS --> CS["Consolidated\nStore"]

    CT -.->|"Trigger conditions:\n- Size threshold\n- Time interval\n- Manual"| CE

    style MS fill:#5a4a2d,stroke:#a84,color:#fff
    style CS fill:#2d5a2d,stroke:#4a9,color:#fff
    style CR fill:#5a2d4a,stroke:#a49,color:#fff
```

</details>

**数据流**：条件满足时（记忆库大小、经过的时间或手动请求），**巩固触发条件**生效。

**聚类引擎**按语义相似度给记忆分组，得到一批相关记忆的簇。

在每个簇内，**合并与去重**模块把重叠的条目合并起来，删除近似重复项。**矛盾解决器**按照配置的策略（新近度、来源优先或 LLM 裁决）处理相互矛盾的记忆。

**重要性打分器**为所有幸存的记忆重新计算分数。最终得到一个干净的**巩固后记忆库**，替换原来的版本。


## 准备工作

安装依赖并配置 API 访问。


In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai

导入 OpenAI SDK 和标准库辅助工具。API 密钥从 `.env` 文件加载。


In [ ]:
import os
import time
import uuid
from dataclasses import dataclass, field
import numpy as np

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们将分五部分搭建巩固流程：

1. **记忆数据模型**与存储。
2. **嵌入辅助函数**，用于计算相似度。
3. **聚类引擎**，把相关记忆分组。
4. **合并、去重与矛盾解决**模块。
5. **重要性打分器**与主**巩固器**类。


### 记忆数据模型

每条记忆除了内容还带有元数据。`source` 字段记录记忆来自哪里：用户直接说的、智能体推断的，还是工具产出的。我们同时记录创建时间、访问次数和重要性分数。


In [ ]:
@dataclass
class Memory:
    content: str
    source: str = "user"  # "user", "inferred", or "tool"
    created_at: float = field(default_factory=time.time)
    last_accessed: float = field(default_factory=time.time)
    access_count: int = 1
    importance: float = 0.5
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    embedding: list[float] | None = None

    def __repr__(self):
        preview = self.content[:60] + ("..." if len(self.content) > 60 else "")
        return f"Memory(id={self.id[:8]}, importance={self.importance:.2f}, content='{preview}')"

### 记忆库（Memory Store）

一个字典的薄封装。它把记忆 ID 映射到 Memory 对象，支持批量替换。巩固器用它换入清理后的版本。


In [ ]:
class MemoryStore:
    def __init__(self):
        self.memories: dict[str, Memory] = {}

    def add(self, memory: Memory) -> None:
        self.memories[memory.id] = memory

    def get_all(self) -> list[Memory]:
        return list(self.memories.values())

    def remove(self, memory_id: str) -> None:
        self.memories.pop(memory_id, None)

    def replace_all(self, memories: list[Memory]) -> None:
        self.memories = {m.id: m for m in memories}

    def __len__(self) -> int:
        return len(self.memories)

    def __repr__(self) -> str:
        return f"MemoryStore({len(self)} memories)"

### 嵌入与相似度辅助函数

我们需要一种方法衡量两条记忆在含义上有多相似。OpenAI 嵌入 API 把文本转换成向量（一串数字），再用余弦相似度比较向量。分数接近 1.0 表示两段文本非常相似。


In [ ]:

def ensure_embeddings(memories: list[Memory]) -> None:
    """为还没有嵌入的记忆计算嵌入。"""
    needs_embedding = [m for m in memories if m.embedding is None]
    if not needs_embedding:
        return
    texts = [m.content for m in needs_embedding]
    vectors = embed_texts(texts)
    for memory, vector in zip(needs_embedding, vectors):
        memory.embedding = vector


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个嵌入向量之间的余弦相似度。"""
    a_arr, b_arr = np.array(a), np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))

### 聚类引擎

把聚类想象成把文件分堆摆到桌上。你每拿起一对就问：「这两份是同一个主题吗？」如果是，就放进同一堆。

聚类引擎用余弦相似度比较每一对记忆。如果两条记忆的分数超过阈值（默认 0.80），它们就归入同一个簇。我们用并查集（union-find，一种按传递关系分组的数据结构）来高效构建簇。


In [ ]:
class ClusterEngine:
    """按嵌入相似度给相关记忆分组。"""

    def __init__(self, similarity_threshold: float = 0.80):
        self.similarity_threshold = similarity_threshold

    def cluster(self, memories: list[Memory]) -> list[list[Memory]]:
        """用并查集按余弦相似度把记忆分组成簇。"""
        ensure_embeddings(memories)
        n = len(memories)

        # 并查集：parent[i] 指向 i 所在组的代表元素
        parent = list(range(n))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]  # 路径压缩
                x = parent[x]
            return x

        def union(x: int, y: int) -> None:
            px, py = find(x), find(y)
            if px != py:
                parent[px] = py

        # 比较每一对；相似度超过阈值就合并
        for i in range(n):
            for j in range(i + 1, n):
                sim = cosine_similarity(memories[i].embedding, memories[j].embedding)
                if sim >= self.similarity_threshold:
                    union(i, j)

        # 收集簇
        clusters: dict[int, list[Memory]] = {}
        for i in range(n):
            root = find(i)
            clusters.setdefault(root, []).append(memories[i])

        return list(clusters.values())

### 合并与去重

在一个簇内，有些记忆可能是近似重复项（相似度高于 0.92）。我们先找出这些组，再让 LLM 把它们合并成一条更丰富的条目。这样能保留每条重复项里最好的细节。


In [ ]:
def find_duplicates(cluster: list[Memory], threshold: float = 0.92) -> list[list[Memory]]:
    """在簇内找出近似重复的记忆组。"""
    n = len(cluster)
    if n <= 1:
        return []

    parent = list(range(n))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x: int, y: int) -> None:
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py

    for i in range(n):
        for j in range(i + 1, n):
            sim = cosine_similarity(cluster[i].embedding, cluster[j].embedding)
            if sim >= threshold:
                union(i, j)

    groups: dict[int, list[Memory]] = {}
    for i in range(n):
        root = find(i)
        groups.setdefault(root, []).append(cluster[i])

    # 只返回含 2 条及以上条目的组（真正的重复项）
    return [g for g in groups.values() if len(g) > 1]

找到重复组之后，我们让 LLM 来合并。模型会看到所有版本及其来源和日期，然后写出一条保留各方最具体细节的新条目。


In [ ]:
def merge_memories_with_llm(duplicates: list[Memory]) -> str:
    """让 LLM 把重复记忆合并成一条干净的条目。"""
    numbered = "\n".join(
        f"{i+1}. [{m.source}, {time.strftime('%Y-%m-%d', time.localtime(m.created_at))}] {m.content}"
        for i, m in enumerate(duplicates)
    )

    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You merge duplicate memory entries into one concise entry. "
                    "Combine all unique information. Keep the most specific details. "
                    "Return only the merged text, nothing else."
                ),
            },
            {
                "role": "user",
                "content": f"Merge these duplicate memories:\n{numbered}",
            },
        ])

    content = to_text(response.content)
    return content.strip() if content else duplicates[0].content

### 矛盾检测与解决

重复项说的是同一件事，矛盾项说的是相反的事。「用户偏好 Python」和「用户换用了 Rust」不可能同时成立。

我们用 LLM 检测簇内的矛盾，然后由 `ConflictResolver` 用三种策略之一选出胜者：

- **新近度（recency）**：最新的记忆获胜。
- **来源优先（source priority）**：用户直接陈述的事实胜过推断，推断胜过工具输出。
- **LLM 裁决（LLM-adjudicated）**：让模型挑选最可能正确的版本。


In [ ]:
def detect_contradictions(cluster: list[Memory]) -> list[list[Memory]]:
    """用 LLM 找出簇内相互矛盾的记忆。"""
    if len(cluster) <= 1:
        return []

    numbered = "\n".join(
        f"{i+1}. {m.content}" for i, m in enumerate(cluster)
    )

    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You detect contradictions between memory entries. "
                    "List pairs of entry numbers that contradict each other. "
                    "Format: one pair per line as 'X,Y' (numbers only). "
                    "If no contradictions exist, reply with 'NONE'."
                ),
            },
            {
                "role": "user",
                "content": f"Find contradictions:\n{numbered}",
            },
        ])

    text = to_text(response.content) or ""
    text = text.strip()
    if text == "NONE" or not text:
        return []

    conflict_groups = []
    for line in text.split("\n"):
        line = line.strip()
        if "," not in line:
            continue
        try:
            parts = [int(x.strip()) - 1 for x in line.split(",")]
            group = [cluster[i] for i in parts if 0 <= i < len(cluster)]
            if len(group) >= 2:
                conflict_groups.append(group)
        except (ValueError, IndexError):
            continue

    return conflict_groups

`ConflictResolver` 从相互矛盾的记忆中选出胜者，支持三种策略。
「新近度」保留最新的记忆。「来源优先」把用户直接陈述的事实排在推断之上。「LLM」让模型挑选最可信的版本。


In [ ]:
class ConflictResolver:
    """从一组矛盾记忆中选出权威的一条。"""

    def __init__(self, strategy: str = "recency"):
        self.strategy = strategy

    def resolve(self, memories: list[Memory]) -> Memory:
        if self.strategy == "recency":
            return self._recency_wins(memories)
        elif self.strategy == "source_priority":
            return self._source_priority(memories)
        elif self.strategy == "llm":
            return self._llm_adjudicate(memories)
        raise ValueError(f"Unknown strategy: {self.strategy}")

    def _recency_wins(self, memories: list[Memory]) -> Memory:
        return max(memories, key=lambda m: m.created_at)

    def _source_priority(self, memories: list[Memory]) -> Memory:
        priority = {"user": 3, "inferred": 2, "tool": 1}
        return max(memories, key=lambda m: (priority.get(m.source, 0), m.created_at))

    def _llm_adjudicate(self, memories: list[Memory]) -> Memory:
        numbered = "\n".join(
            f"{i+1}. [{m.source}, {time.strftime('%Y-%m-%d', time.localtime(m.created_at))}] {m.content}"
            for i, m in enumerate(memories)
        )

        response = llm.invoke([
                {
                    "role": "system",
                    "content": (
                        "You resolve contradictory memory entries. "
                        "Pick the entry that is most likely correct and current. "
                        "Consider recency and source reliability. "
                        "Reply with ONLY the entry number."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Which memory is most likely correct?\n{numbered}",
                },
            ])

        try:
            text = to_text(response.content) or ""
            chosen_idx = int(text.strip()) - 1
            return memories[max(0, min(chosen_idx, len(memories) - 1))]
        except (ValueError, IndexError):
            return self._recency_wins(memories)

### 重要性打分

合并和矛盾解决之后，我们重新计算每条记忆的重要性。公式对三个因子加权：

- **新近度**（40%）：越新的记忆分数越高。我们使用指数衰减。
- **访问频率**（30%）：访问越频繁的记忆越有价值。
- **来源可靠性**（30%）：用户直接陈述的事实分数最高。


In [ ]:
def rescore_importance(memories: list[Memory], decay_rate: float = 0.01) -> None:
    """重新计算所有记忆的重要性分数。"""
    now = time.time()
    source_weights = {"user": 1.0, "inferred": 0.6, "tool": 0.4}
    max_access = max((m.access_count for m in memories), default=1)

    for memory in memories:
        # 指数时间衰减（值越高表示越新）
        age_hours = (now - memory.created_at) / 3600
        recency = float(np.exp(-decay_rate * age_hours))

        # 归一化的访问频率
        frequency = memory.access_count / max_access

        # 来源可靠性
        source_w = source_weights.get(memory.source, 0.5)

        memory.importance = round(
            0.4 * recency + 0.3 * frequency + 0.3 * source_w, 3
        )

### 巩固器（Consolidator）

这个类统筹整个流程。它检查触发条件，聚类记忆，合并重复项，解决矛盾，重新打重要性分数，并修剪低价值条目。


In [ ]:
@dataclass
class ConsolidationReport:
    memories_before: int = 0
    memories_after: int = 0
    duplicates_merged: int = 0
    conflicts_resolved: int = 0
    clusters_found: int = 0


class MemoryConsolidator:
    """运行完整的记忆巩固流程。"""

    def __init__(
        self,
        store: MemoryStore,
        similarity_threshold: float = 0.80,
        duplicate_threshold: float = 0.92,
        conflict_strategy: str = "recency",
        size_trigger: int = 10,
        prune_below: float = 0.15,
    ):
        self.store = store
        self.clusterer = ClusterEngine(similarity_threshold)
        self.resolver = ConflictResolver(conflict_strategy)
        self.duplicate_threshold = duplicate_threshold
        self.size_trigger = size_trigger
        self.prune_below = prune_below
        self.last_run: float = 0.0

    def should_consolidate(self) -> bool:
        """检查记忆库是否达到大小触发条件。"""
        return len(self.store) >= self.size_trigger

`consolidate` 方法按顺序运行完整流程：为所有记忆生成嵌入，聚类，在每个簇内合并重复项，解决矛盾，重新打分，修剪低价值条目。它返回一份汇总变化的报告。


巩固循环逐个处理每个簇。这个辅助函数处理单个簇。
它合并近似重复项（几乎完全相同的记忆），并解决矛盾（意见相左的记忆）。
把这段逻辑抽成独立的函数，能让每个部分保持简短、可测试。


In [ ]:
def _process_single_cluster(cluster, duplicate_threshold, resolver):
    """处理一个簇：合并重复项并解决矛盾。

    返回 (幸存记忆, 被合并的重复项数, 被解决的矛盾数)。
    """
    working = list(cluster)
    dup_count = 0
    conflict_count = 0

    # 合并近似重复项
    dup_groups = find_duplicates(working, duplicate_threshold)
    for dup_group in dup_groups:
        merged_text = merge_memories_with_llm(dup_group)
        newest = max(dup_group, key=lambda m: m.created_at)
        merged = Memory(
            content=merged_text,
            source=newest.source,
            created_at=newest.created_at,
            last_accessed=max(m.last_accessed for m in dup_group),
            access_count=sum(m.access_count for m in dup_group),
            importance=max(m.importance for m in dup_group),
        )
        dup_ids = {m.id for m in dup_group}
        working = [m for m in working if m.id not in dup_ids]
        working.append(merged)
        dup_count += len(dup_group)

    # 检测并解决矛盾
    conflicts = detect_contradictions(working)
    for conflict_group in conflicts:
        winner = resolver.resolve(conflict_group)
        loser_ids = {m.id for m in conflict_group if m.id != winner.id}
        working = [m for m in working if m.id not in loser_ids]
        conflict_count += len(loser_ids)

    return working, dup_count, conflict_count

现在定义主方法 `consolidate`。它为所有记忆生成嵌入并聚类，
然后对每个簇调用 `_process_single_cluster`。处理完成后，重新计算重要性分数，
并修剪低价值条目。


In [ ]:
def consolidate(self) -> ConsolidationReport:
    """运行一轮完整的巩固。"""
    report = ConsolidationReport()
    memories = self.store.get_all()
    report.memories_before = len(memories)

    if len(memories) <= 1:
        report.memories_after = len(memories)
        return report

    # 第 1 步：为所有记忆生成嵌入
    ensure_embeddings(memories)

    # 第 2 步：按相似度聚类
    clusters = self.clusterer.cluster(memories)
    report.clusters_found = len(clusters)

    consolidated = []

    for cluster in clusters:
        if len(cluster) == 1:
            consolidated.append(cluster[0])
            continue

        working, dup_count, conflict_count = _process_single_cluster(
            cluster, self.duplicate_threshold, self.resolver,
        )
        report.duplicates_merged += dup_count
        report.conflicts_resolved += conflict_count
        consolidated.extend(working)

    # 第 5 步：重新计算重要性分数
    rescore_importance(consolidated)

    # 第 6 步：修剪低重要性记忆
    consolidated = [m for m in consolidated if m.importance >= self.prune_below]

    # 为新合并出的记忆重新计算嵌入
    ensure_embeddings(consolidated)

    self.store.replace_all(consolidated)
    self.last_run = time.time()
    report.memories_after = len(consolidated)

    return report


MemoryConsolidator.consolidate = consolidate

## 运行示例

我们来看看巩固的实际效果。先往记忆库里塞入杂乱的数据：重复项、矛盾和噪声。然后运行巩固器，对比前后差异。


首先，我们创建 12 条智能体几天里可能积累的记忆。有些是近似重复项（三条关于用户职业的条目），有些互相矛盾（Python 与 Rust 的偏好之争），还有一条是低重要性噪声。


In [ ]:
store = MemoryStore()

day = 86400  # 一天的秒数
now = time.time()

messy_memories = [
    # 三条关于用户职业的近似重复记忆
    Memory(
        content="User works as a software engineer at Acme Corp",
        source="user", created_at=now - 7 * day, access_count=3,
    ),
    Memory(
        content="The user is a software engineer working at Acme Corp",
        source="inferred", created_at=now - 5 * day, access_count=1,
    ),
    Memory(
        content="User's occupation: software engineer, employer: Acme Corp",
        source="tool", created_at=now - 3 * day, access_count=2,
    ),
    # 相互矛盾的编程语言偏好
    Memory(
        content="User prefers Python for most programming tasks",
        source="user", created_at=now - 10 * day, access_count=5,
    ),
    Memory(
        content="User recently switched to Rust as their primary language",
        source="user", created_at=now - 1 * day, access_count=2,
    ),
]

我们再往测试集里加几条记忆：项目相关的重复项、独立的事实、一条地点重复项，以及一条低重要性噪声。这些给巩固器提供了一组贴近现实的混合数据来清理。


In [ ]:
more_memories = [
    # 两条关于一个项目的近似重复记忆
    Memory(
        content="User is building a chatbot with memory for their startup",
        source="user", created_at=now - 6 * day, access_count=4,
    ),
    Memory(
        content="User's current project: an AI chatbot that remembers conversations",
        source="inferred", created_at=now - 4 * day, access_count=1,
    ),
    # 独立的记忆（应原样保留）
    Memory(
        content="User has a golden retriever named Max",
        source="user", created_at=now - 8 * day, access_count=2,
    ),
    Memory(
        content="User lives in San Francisco",
        source="user", created_at=now - 9 * day, access_count=3,
    ),
    Memory(
        content="User prefers morning meetings before 10 AM",
        source="inferred", created_at=now - 2 * day, access_count=1,
    ),
    # 地点记忆的近似重复项
    Memory(
        content="User is based in San Francisco, California",
        source="user", created_at=now - 3 * day, access_count=2,
    ),
    # 低重要性噪声
    Memory(
        content="User mentioned the weather was nice today",
        source="inferred", created_at=now - 14 * day, access_count=1,
        importance=0.1,
    ),
]

messy_memories.extend(more_memories)

现在把全部 12 条记忆装入记忆库并打印出来。注意其中的重复项、矛盾和噪声。


In [ ]:

for m in messy_memories:
    store.add(m)

print(f"Memory store before consolidation: {len(store)} entries\n")
for m in store.get_all():
    print(f"  [{m.source:8s}] {m.content}")

现在运行巩固。巩固器会聚类这些记忆，合并重复项，解决 Python 与 Rust 的矛盾，重新打重要性分数，并修剪那条关于天气的噪声。


In [ ]:
consolidator = MemoryConsolidator(
    store=store,
    similarity_threshold=0.80,
    duplicate_threshold=0.90,
    conflict_strategy="recency",
    size_trigger=5,
    prune_below=0.15,
)

print(f"Should consolidate? {consolidator.should_consolidate()}")
print("Running consolidation...\n")

report = consolidator.consolidate()

print(f"Memories before:      {report.memories_before}")
print(f"Memories after:       {report.memories_after}")
print(f"Clusters found:       {report.clusters_found}")
print(f"Duplicates merged:    {report.duplicates_merged}")
print(f"Conflicts resolved:   {report.conflicts_resolved}")

检查巩固后的记忆库。你应该会看到条目更少、没有重复项，Python 偏好已被 Rust 取代。


In [ ]:
print(f"Consolidated store: {len(store)} entries\n")

for m in sorted(store.get_all(), key=lambda m: m.importance, reverse=True):
    print(f"  [importance={m.importance:.3f}] [{m.source:8s}] {m.content}")

## 权衡

### 巩固效果好的场景

- **长期运行的智能体**：跨会话积累几百条记忆。定期清理让记忆库保持精简、准确。
- **事实准确性很重要**：相互矛盾的记忆会导致错误回答。巩固在矛盾到达用户之前就把它解决掉。
- **存储与检索成本**：记忆更少，相似度搜索更快，嵌入存储成本更低。
- **检索质量**：合并、去重后的记忆更完整。智能体检索到的是一条丰富的条目，而不是三条残缺的。

### 巩固失效的场景

- **设计上就有损**：合并和修剪可能丢弃以后有用的信息。被修剪的记忆无法找回。
- **LLM 调用带来成本**：合并和矛盾解决都需要 API 调用。一个有一万条记忆的库，每轮巩固可能需要几百次 LLM 调用。
- **聚类错误**：相似度阈值太低，无关记忆会被合并；太高，真正的重复项会漏网。调参取决于你的领域。
- **时间维度的细微差别会丢失**：「用户偏好 Python」和「用户换用了 Rust」不一定是矛盾。用户可能用 Python 写脚本、用 Rust 做系统开发。巩固可能把问题过度简化。
- **巩固期间的延迟**：巩固过程运行时会拖慢智能体。生产系统会异步运行它，或安排在空闲时段运行。


## 延伸阅读

- Stickgold & Walker, ["Sleep-Dependent Memory Triage: Evolving Generalization through Selective Processing,"](https://doi.org/10.1038/nn.3303) *Nature Neuroscience*, 2013。探讨大脑如何在睡眠中选择性地巩固重要记忆。
- Packer et al., ["MemGPT: Towards LLMs as Operating Systems,"](https://arxiv.org/abs/2310.08560) 2023。在一个受操作系统启发的 LLM 架构中实现记忆管理，包括类似巩固的操作。
- Rasch & Born, ["About Sleep's Role in Memory,"](https://doi.org/10.1152/physrev.00032.2012) *Physiological Reviews*, 2013。神经科学领域关于睡眠记忆巩固的全面综述。
- Zhong et al., ["MemoryBank: Enhancing Large Language Models with Long-Term Memory,"](https://arxiv.org/abs/2305.10250) 2024。实现了遗忘曲线和记忆更新，与巩固过程异曲同工。


---

*← 上一篇：[13 - 分层记忆](../13_hierarchical_memory_layers/) · 下一篇：[15 - 记忆压缩](../15_memory_compaction/) →*


## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：聚类阈值扫描
把 `ClusterEngine` 的相似度阈值从 0.70 到 0.95 以 0.05 为步长变化。对每个阈值，在同一个记忆集上运行巩固，记录簇的数量、被合并的记忆数和孤立记忆数。画出阈值与簇数量的关系图。

### 挑战 2：前后指标对比
在运行 `MemoryConsolidator` 之前和之后，测量：记忆总数、token 总数（所有记忆内容长度之和）、检测到的矛盾数量、近似重复项数量。给出一张清晰的前后对比表。

### 挑战 3：对话中的周期性巩固
把 `MemoryConsolidator` 包进一个循环，每 10 个对话轮次触发一次巩固。跟踪 50 轮对话中记忆数量、矛盾率和检索准确率的变化。与 19 遗忘与衰减中的基于衰减的方法做比较。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--14-memory-consolidation--memory-consolidation)